In [65]:
import pandas as pd 
import numpy as np

from matplotlib import pyplot as plt
from sklearn.preprocessing import StandardScaler

import datetime

import ipympl 
%matplotlib ipympl


file_path = 'ANN_final_model_pred_3-2022_3-2023_national_holiday.csv'

df = pd.read_csv(file_path)

df.set_index('Date Time', inplace=True)

df.index = pd.to_datetime(df.index)

# df['month'] = df.index.month
# df['weekday'] = df.index.weekday

# Drop the specified columns
df.drop(columns=['Predicted kW'], inplace=True)

df_1hr = df.resample('1H').mean()

display(df)
display(df_1hr)

,Real kW
Date Time,
2022-03-24 00:00:00,121.500
2022-03-24 00:15:00,121.500
2022-03-24 00:30:00,119.248
2022-03-24 00:45:00,120.600
2022-03-24 01:00:00,123.300
...,...
2023-03-22 22:45:00,117.448
2023-03-22 23:00:00,115.200
2023-03-22 23:15:00,118.348


,Real kW
Date Time,
2022-03-24 00:00:00,120.712
2022-03-24 01:00:00,118.575
2022-03-24 02:00:00,124.088
2022-03-24 03:00:00,194.963
2022-03-24 04:00:00,222.526
...,...
2023-03-22 19:00:00,123.863
2023-03-22 20:00:00,117.674
2023-03-22 21:00:00,118.125


In [66]:
mask = (df.index.month >= 6) & (df.index.month < 7)  # June
mask_1hr = (df_1hr.index.month >= 6) & (df_1hr.index.month < 7)

mask2 = (df.index.month >= 7) & (df.index.month < 8)  # July
mask2_1hr = (df_1hr.index.month >= 7) & (df_1hr.index.month < 8)

mask3 = (df.index.month >=  8) & (df.index.month < 9) # August
mask3_1hr = (df_1hr.index.month >= 8) & (df_1hr.index.month < 9)

# mask4 = (df.index.month >=  9) & (df.index.month < 10) # September

df_june = df[mask]
df_1hr_june = df_1hr[mask_1hr]

df_july = df[mask2]
df_1hr_july = df_1hr[mask2_1hr]

df_august = df[mask3]
df_1hr_august = df_1hr[mask3_1hr]

# df_september = df[mask4]

display(df_june)
display(df_1hr_june)
# display(df_july)
# display(df_august)
# display(df_september)

,Real kW
Date Time,
2022-06-01 00:00:00,109.800
2022-06-01 00:15:00,110.248
2022-06-01 00:30:00,110.248
2022-06-01 00:45:00,109.348
2022-06-01 01:00:00,110.700
...,...
2022-06-30 22:45:00,121.500
2022-06-30 23:00:00,118.800
2022-06-30 23:15:00,116.552


,Real kW
Date Time,
2022-06-01 00:00:00,109.911
2022-06-01 01:00:00,110.812
2022-06-01 02:00:00,112.050
2022-06-01 03:00:00,110.924
2022-06-01 04:00:00,117.900
...,...
2022-06-30 19:00:00,123.188
2022-06-30 20:00:00,121.613
2022-06-30 21:00:00,121.725


In [68]:
#   input a month of data with 15 min and 1hr intervals, flat fee, on-peak electric charge rate, off-peak electric charge rate and demand charge
# 
#   Bill            =      Flat Fee        +     Electric Charge     +   Demand Charge 
#   Electric Charge = (On-Peak Rate * KWh (12:00 PM to 6 PM MDT Monday - friday)) + (Off-Peak Rate * kWh (hours not covered in on-peak period))
#   Demand Charge   = (Highest Demand in a single 15-min interval * Demand Charge Rate)       

def calculate_utility_bill(data, data_1hr, flat_fee, on_peak_rate, off_peak_rate, demand_charge_rate):
    
    # Calculate electric charge
    on_peak_hours = data_1hr[(data_1hr.index.weekday < 5) & (data_1hr.index.hour >= 12) & (data_1hr.index.hour < 18)]
    on_peak_hours_sum = data_1hr[(data_1hr.index.weekday < 5) & (data_1hr.index.hour >= 12) & (data_1hr.index.hour < 18)].sum()  # Sum of kWh during on-peak hours
    
    off_peak_hours = data_1hr.drop(on_peak_hours.index).sum()  # Sum of kWh during off-peak hours
    # off_peak_hours = data_1hr[(data_1hr.index.weekday < 5) & (data_1hr.index.hour >= 12) & (data_1hr.index.hour < 18)].sum()  # Sum of kWh during off-peak hours

    electric_charge = (on_peak_rate * on_peak_hours_sum) + (off_peak_rate * off_peak_hours)
    
    # Calculate demand charge
    highest_demand = data.max()  # Highest demand in a single 15-min interval
    demand_charge = highest_demand * demand_charge_rate
    
    # Calculate total bill
    total_bill = flat_fee + electric_charge + demand_charge
    
    return total_bill

In [69]:
# Set parameters
flat_fee = 30.20  # Flat fee
on_peak_rate = 0.11861  # On-peak electric charge rate
off_peak_rate = 0.00502  # Off-peak electric charge rate
demand_charge_rate = 24.5  # Demand charge rate ($ per kWh)

# Calculate utility bill
utility_bill = calculate_utility_bill(df_june, df_1hr_june, flat_fee, on_peak_rate, off_peak_rate, demand_charge_rate)
print("Total utility bill in June:", utility_bill.values[0])

utility_bill2 = calculate_utility_bill(df_july, df_1hr_july, flat_fee, on_peak_rate, off_peak_rate, demand_charge_rate)
print("Total utility bill in July:", utility_bill2.values[0])

utility_bill3 = calculate_utility_bill(df_august, df_1hr_august, flat_fee, on_peak_rate, off_peak_rate, demand_charge_rate)
print("Total utility bill in August:", utility_bill3.values[0])

Total utility bill in June: 20720.598097609996
Total utility bill in July: 21391.158209189998
Total utility bill in August: 20138.06910461
